In [ ]:
#ataset Link = https://github.com/DataScience75/Top_mentor_projects_Datasets/blob/main/customer_reviews.csv
#Groq URL = https://console.groq.com/playground

In [ ]:
!pip install faiss-cpu groq python-dotenv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 35.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 6.7 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer
from groq import Groq
from sklearn.metrics import precision_score
from dotenv import load_dotenv
import os
import re

In [ ]:
groq_api_key = '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'
client = Groq(api_key=groq_api_key)

In [ ]:
df = pd.read_csv('/content/iphone.csv')
df.head(5)

,productAsin,country,date,isVerified,ratingScore,reviewTitle,reviewDescription,reviewUrl,reviewedIn,variant,variantAsin
0,B09G9BL5CP,India,11-08-2024,True,4,No charger,"Every thing is good about iPhones, there's not...",https://www.amazon.in/gp/customer-reviews/R345...,Reviewed in India on 11 August 2024,Colour: MidnightSize: 256 GB,B09G9BQS98
1,B09G9BL5CP,India,16-08-2024,True,5,iPhone 13 256GB,"It look so fabulous, I am android user switche...",https://www.amazon.in/gp/customer-reviews/R2HJ...,Reviewed in India on 16 August 2024,Colour: MidnightSize: 256 GB,B09G9BQS98
2,B09G9BL5CP,India,14-05-2024,True,4,Flip camera option nill,I tried to flip camera while recording but no ...,https://www.amazon.in/gp/customer-reviews/R3Y7...,Reviewed in India on 14 May 2024,Colour: MidnightSize: 256 GB,B09G9BQS98
3,B09G9BL5CP,India,24-06-2024,True,5,Product,100% genuine,https://www.amazon.in/gp/customer-reviews/R1P9...,Reviewed in India on 24 June 2024,Colour: MidnightSize: 256 GB,B09G9BQS98
4,B09G9BL5CP,India,18-05-2024,True,5,Good product,Happy to get the iPhone 13 in Amazon offer,https://www.amazon.in/gp/customer-reviews/R1XI...,Reviewed in India on 18 May 2024,Colour: MidnightSize: 256 GB,B09G9BQS98


In [ ]:
def preprocess_text(text):
  text = re.sub(r"\s+"," ",text)
  text = re.sub(r"[^a-zA-Z0-9 ]","",text)
  return text.strip().lower()

In [ ]:
df['reviewDescription'].astype("str").apply(preprocess_text)

,reviewDescription
0,every thing is good about iphones theres nothi...
1,it look so fabulous i am android user switched...
2,i tried to flip camera while recording but no ...
3,100 genuine
4,happy to get the iphone 13 in amazon offer
...,...
3057,useless phon never buy this heat n useless cam...
3058,iam not happy with this product why because ch...
3059,good phone
3060,while charging mobile its getting so hot even ...


In [ ]:
df['clean_text'] = df['reviewDescription'].astype("str").apply(preprocess_text)
df

,productAsin,country,date,isVerified,ratingScore,reviewTitle,reviewDescription,reviewUrl,reviewedIn,variant,variantAsin,clean_text
0,B09G9BL5CP,India,11-08-2024,True,4,No charger,"Every thing is good about iPhones, there's not...",https://www.amazon.in/gp/customer-reviews/R345...,Reviewed in India on 11 August 2024,Colour: MidnightSize: 256 GB,B09G9BQS98,every thing is good about iphones theres nothi...
1,B09G9BL5CP,India,16-08-2024,True,5,iPhone 13 256GB,"It look so fabulous, I am android user switche...",https://www.amazon.in/gp/customer-reviews/R2HJ...,Reviewed in India on 16 August 2024,Colour: MidnightSize: 256 GB,B09G9BQS98,it look so fabulous i am android user switched...
2,B09G9BL5CP,India,14-05-2024,True,4,Flip camera option nill,I tried to flip camera while recording but no ...,https://www.amazon.in/gp/customer-reviews/R3Y7...,Reviewed in India on 14 May 2024,Colour: MidnightSize: 256 GB,B09G9BQS98,i tried to flip camera while recording but no ...
3,B09G9BL5CP,India,24-06-2024,True,5,Product,100% genuine,https://www.amazon.in/gp/customer-reviews/R1P9...,Reviewed in India on 24 June 2024,Colour: MidnightSize: 256 GB,B09G9BQS98,100 genuine
4,B09G9BL5CP,India,18-05-2024,True,5,Good product,Happy to get the iPhone 13 in Amazon offer,https://www.amazon.in/gp/customer-reviews/R1XI...,Reviewed in India on 18 May 2024,Colour: MidnightSize: 256 GB,B09G9BQS98,happy to get the iphone 13 in amazon offer
...,...,...,...,...,...,...,...,...,...,...,...,...
3057,B09G9D8KRQ,India,11-10-2023,True,1,Very bad experience with i phone 13,Useless phon never buy this heat n useless cam...,https://www.amazon.in/gp/customer-reviews/R10O...,Reviewed in India on 11 October 2023,Colour: (PRODUCT) REDSize: 128 GB,B09G99CW2N,useless phon never buy this heat n useless cam...
3058,B09G9D8KRQ,India,14-10-2022,True,2,not happy with this apple product,iam not happy with this product why because ch...,https://www.amazon.in/gp/customer-reviews/R2FW...,Reviewed in India on 14 October 2022,Colour: (PRODUCT) REDSize: 128 GB,B09G99CW2N,iam not happy with this product why because ch...
3059,B09G9D8KRQ,India,24-02-2022,True,3,Good phone,Good phone,https://www.amazon.in/gp/customer-reviews/R2C7...,Reviewed in India on 24 February 2022,Colour: (PRODUCT) REDSize: 128 GB,B09G99CW2N,good phone
3060,B09G9D8KRQ,India,16-10-2023,True,1,Battery discharge,While charging mobile it's getting so hot even...,https://www.amazon.in/gp/customer-reviews/R3K0...,Reviewed in India on 16 October 2023,Colour: (PRODUCT) REDSize: 128 GB,B09G99CW2N,while charging mobile its getting so hot even ...


In [ ]:
def chunk_text(text,chunk_size=20):
  words = text.split()
  return[" ".join(words[i:i+chunk_size]) for i in range(0,len(words),chunk_size)]

In [ ]:
chunks = []
metadata = []
for idx, row in df.iterrows():
  for clean_chunk in chunk_text(row['clean_text']):
    chunks.append(clean_chunk)
    metadata.append({'product_id':row['productAsin'],'review_id':idx})

In [ ]:
metadata

[{'product_id': 'B09G9BL5CP', 'review_id': 0},
 {'product_id': 'B09G9BL5CP', 'review_id': 0},
 {'product_id': 'B09G9BL5CP', 'review_id': 0},
 {'product_id': 'B09G9BL5CP', 'review_id': 0},
 {'product_id': 'B09G9BL5CP', 'review_id': 1},
 {'product_id': 'B09G9BL5CP', 'review_id': 1},
 {'product_id': 'B09G9BL5CP', 'review_id': 2},
 {'product_id': 'B09G9BL5CP', 'review_id': 2},
 {'product_id': 'B09G9BL5CP', 'review_id': 2},
 {'product_id': 'B09G9BL5CP', 'review_id': 2},
 {'product_id': 'B09G9BL5CP', 'review_id': 2},
 {'product_id': 'B09G9BL5CP', 'review_id': 3},
 {'product_id': 'B09G9BL5CP', 'review_id': 4},
 {'product_id': 'B09G9BL5CP', 'review_id': 5},
 {'product_id': 'B09G9BL5CP', 'review_id': 6},
 {'product_id': 'B09G9BL5CP', 'review_id': 6},
 {'product_id': 'B09G9BL5CP', 'review_id': 7},
 {'product_id': 'B09G9BL5CP', 'review_id': 8},
 {'product_id': 'B09G9BL5CP', 'review_id': 9},
 {'product_id': 'B09G9BL5CP', 'review_id': 10},
 {'product_id': 'B09G9BL5CP', 'review_id': 10},
 {'product_

In [ ]:
chunks


['every thing is good about iphones theres nothing compared to its speed and ios but i was disappointed because theres',
 'no charger for it even though indian judiciary told them to provide still they havent made any progress regarding charger',
 'why should we spend 62k and more than that for iphones because of its brand and acessories comfort size and',
 'features right if thats not there whats point of buying an iphonethank you',
 'it look so fabulous i am android user switched to apple performance wise iphone is more better than android due',
 'to a15 bionic chip i feel pleasant while using iphone in hand i like it very much',
 'i tried to flip camera while recording but no facility is added here you have to pause first then you',
 'can flip from back to front or vice versa it means you cant continue it very awkward position iphone software',
 'should be upgraded so that we can make it go its very essential for a content creator i purchased gimbal',
 'dji om se but it could not pr

In [ ]:
model = SentenceTransformer('all-MiniLM-L6-v2')
embeddings = model.encode(chunks,convert_to_numpy= True)
embeddings

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

array([[-0.02696455,  0.02999497,  0.07132509, ..., -0.05796148,
         0.05524072,  0.13675663],
       [-0.08033787,  0.09218963, -0.05183384, ..., -0.03947391,
        -0.01438533,  0.04200704],
       [ 0.00769377,  0.08479298,  0.05187638, ..., -0.12554216,
        -0.0607058 ,  0.06046386],
       ...,
       [-0.13236727,  0.08389042,  0.06909958, ...,  0.02577613,
        -0.01951098,  0.10076311],
       [ 0.01481783,  0.06807026,  0.01756978, ..., -0.01758698,
         0.00928347,  0.04424087],
       [-0.00946623,  0.06474222,  0.01569523, ...,  0.06423365,
        -0.10020681, -0.01162443]], dtype=float32)

In [ ]:
dim = embeddings.shape[1]

In [ ]:
index = faiss.IndexFlatL2(dim)
index.add(embeddings)

In [ ]:
def recommend_products(user_query,top_k = 3):
  query_vec = model.encode([user_query],convert_to_numpy=True)
  distances,indices = index.search(query_vec,top_k)

  retrieved_chunks = [chunks[i] for i in indices[0]]
  retrieved_metadata = [metadata[i] for i in indices[0]]

  context = "\n".join([f"product{m['product_id']}:{txt}" for txt,m in zip(retrieved_chunks,retrieved_metadata)])

  completion = client.chat.completions.create(
    model="llama-3.1-8b-instant",
      messages = [
          {"role":"system","content":"You are helpfull product recomendation assitant"},
          {"role":"user","content":f"Based on the following customer reviews:\n{context}\n\n suggest the best product"}
      ],
      temperature = 0.2,
      max_tokens = 200
  )


  return completion.choices[0].message.content,retrieved_metadata

In [ ]:
recommend_products('I need a mobile phone with good bettery and camera')

('Based on the customer reviews, it seems that there are mixed opinions about the product. However, the majority of the reviews are negative, specifically mentioning the camera performance.\n\nOut of the 3 reviews, 2 of them mention that the camera is "pathetic" and "not good", while the third review is extremely positive, stating that the camera is "amazing".\n\nGiven the conflicting opinions, I would suggest that the product may not be the best choice for someone looking for a good camera phone. However, if you\'re looking for a phone with a great camera, you may want to consider other options, such as the Samsung S23 Plus, which was mentioned in one of the reviews.\n\nIf you could provide more information about what you\'re looking for in a phone, I may be able to suggest a more specific product recommendation.',
 [{'product_id': 'B0BDK8LKPJ', 'review_id': 1432},
  {'product_id': 'B0BDK8LKPJ', 'review_id': 1432},
  {'product_id': 'B0BDK8LKPJ', 'review_id': 1488}])

In [ ]:
test_queries = [('long bettery life','P101'),
                ('fast performance','P103')]

test_queries

[('long bettery life', 'P101'), ('fast performance', 'P103')]

In [ ]:
correct = 0
for query, expected_product in test_queries:
  response,meta = recommend_products(query,top_k=1)
  if meta[0]["product_id"] == expected_product:
    correct += 1

accuracy = correct/len(test_queries)
accuracy

0.0

In [ ]:
!pip install gradio

In [ ]:
import gradio as gr
iface = gr.Interface(fn = recommend_products,
                      inputs = gr.Textbox(lines=2,placeholder="Ask Something...."),
                      outputs = "text",
                      title = "Recommendation Chat bot")
iface.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://919d21bf4db2771a43.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
